In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/df_filtered.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/train.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/test.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/custom.css


In [2]:
import os
for path, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(path, f))

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/df_filtered.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/train.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/test.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/custom.css


In [3]:
import pandas as pd

BASE = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/'

df = pd.read_csv(BASE + 'df_filtered.csv')
train = pd.read_csv(BASE + 'train.csv')
test  = pd.read_csv(BASE + 'test.csv')

print(f"Full filtered : {df.shape}")
print(f"Train         : {train.shape}")
print(f"Test          : {test.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Full filtered : (219515, 6)
Train         : (175802, 6)
Test          : (18083, 6)

Columns: ['UserId', 'ProductId', 'Score', 'Time', 'Summary', 'Text']


In [4]:
# Build one text profile per product
# Combine Summary + Text for richer representation
product_profiles = df.groupby('ProductId').agg({
    'Summary': lambda x: ' '.join(x.dropna().astype(str)),
    'Text'   : lambda x: ' '.join(x.dropna().astype(str))
}).reset_index()

product_profiles['content'] = product_profiles['Summary'] + ' ' + product_profiles['Text']

print(f"Total products with profiles : {len(product_profiles):,}")
print(f"\nSample content (first 200 chars):")
print(product_profiles['content'].iloc[0][:200])

Total products with profiles : 17,538

Sample content (first 200 chars):
so fun to read Children will find it entertaining and a generator of giggles This book is much too small This is my grand daughter's and my favorite book to read. She is 4 and loves the rhythm of this


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import time

tfidf = TfidfVectorizer(
    max_features=10000,  # top 10K most informative terms
    stop_words='english',
    min_df=5,            # term must appear in at least 5 products
    ngram_range=(1, 2)   # unigrams + bigrams
)

start = time.time()
tfidf_matrix = tfidf.fit_transform(product_profiles['content'])
elapsed = time.time() - start

print(f"TF-IDF matrix shape : {tfidf_matrix.shape}")
print(f"Vocabulary size     : {len(tfidf.vocabulary_):,}")
print(f"Time taken          : {elapsed:.1f} seconds")
print(f"Matrix type         : {type(tfidf_matrix)}")
print(f"Sparsity            : {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))*100:.2f}%")

TF-IDF matrix shape : (17538, 10000)
Vocabulary size     : 10,000
Time taken          : 25.5 seconds
Matrix type         : <class 'scipy.sparse._csr.csr_matrix'>
Sparsity            : 97.50%


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Build product index lookup
product_ids = product_profiles['ProductId'].tolist()
product_idx = {pid: idx for idx, pid in enumerate(product_ids)}

def get_similar_products(product_id, n=10):
    if product_id not in product_idx:
        return []
    
    idx = product_idx[product_id]
    
    # Compute similarity only for this one product vs all others
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    # Sort by similarity, exclude the product itself (score=1.0)
    similar_indices = np.argsort(sim_scores)[::-1][1:n+1]
    
    return [(product_ids[i], round(sim_scores[i], 4)) for i in similar_indices]

# Test it on a real product
sample_product = product_profiles['ProductId'].iloc[0]
results = get_similar_products(sample_product, n=10)

print(f"Top 10 products similar to: {sample_product}")
print(f"{'Rank':<6} {'ProductId':<15} {'Similarity'}")
print("-" * 35)
for rank, (pid, score) in enumerate(results, 1):
    print(f"{rank:<6} {pid:<15} {score}")

Top 10 products similar to: 0006641040
Rank   ProductId       Similarity
-----------------------------------
1      B000QUZEBY      0.2205
2      B00206I9RS      0.2066
3      B004VLVIFU      0.1839
4      B000V6L2FK      0.1812
5      B000MIDRKA      0.1611
6      B001228QE2      0.161
7      B0015UW23M      0.1594
8      B000FDQV46      0.1594
9      B00061EPKE      0.1594
10     B0000DIYHW      0.1583


In [7]:
def precision_recall_at_k(test_df, product_idx, tfidf_matrix, product_ids, k=10):
    precisions = []
    recalls = []
    
    # Group test by user
    user_groups = test_df.groupby('UserId')
    
    for user_id, user_test in user_groups:
        # Relevant = products user rated >= 4 in test
        relevant = set(user_test[user_test['Score'] >= 4]['ProductId'].tolist())
        if not relevant:
            continue
        
        # Seed = products user interacted with in train
        user_train = train[train['UserId'] == user_id]['ProductId'].tolist()
        if not user_train:
            continue
        
        # Get recommendations based on user's last rated product
        seed_product = user_train[-1]
        if seed_product not in product_idx:
            continue
        
        recs = [pid for pid, _ in get_similar_products(seed_product, n=k)]
        
        hits = len(set(recs) & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant) if relevant else 0)
    
    return np.mean(precisions), np.mean(recalls), len(precisions)

print("Computing Precision@10 and Recall@10 on test set...")
print("(This will take a few minutes...)")

precision, recall, n_users = precision_recall_at_k(test, product_idx, tfidf_matrix, product_ids, k=10)

print(f"\nContent-Based Results")
print(f"=====================")
print(f"Precision@10 : {precision:.4f}")
print(f"Recall@10    : {recall:.4f}")
print(f"Users evaluated : {n_users:,}")

Computing Precision@10 and Recall@10 on test set...
(This will take a few minutes...)

Content-Based Results
Precision@10 : 0.0096
Recall@10    : 0.0252
Users evaluated : 3,876


In [8]:
# Coverage = what % of catalog appears in recommendations across all users
sample_users = train['UserId'].unique()[:500]  # sample for speed
all_recs = set()

for user_id in sample_users:
    user_products = train[train['UserId'] == user_id]['ProductId'].tolist()
    if not user_products:
        continue
    seed = user_products[-1]
    if seed not in product_idx:
        continue
    recs = [pid for pid, _ in get_similar_products(seed, n=10)]
    all_recs.update(recs)

coverage = len(all_recs) / len(product_ids)

print(f"Unique products recommended : {len(all_recs):,}")
print(f"Total products in catalog   : {len(product_ids):,}")
print(f"Coverage                    : {coverage*100:.2f}%")

Unique products recommended : 1,651
Total products in catalog   : 17,538
Coverage                    : 9.41%


In [9]:
import pickle

# Save TF-IDF vectorizer and matrix
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

import scipy.sparse as sp
sp.save_npz('tfidf_matrix.npz', tfidf_matrix)

product_profiles.to_csv('product_profiles.csv', index=False)

print("Saved:")
print(f"  tfidf_vectorizer.pkl")
print(f"  tfidf_matrix.npz")
print(f"  product_profiles.csv : {len(product_profiles):,} rows")

Saved:
  tfidf_vectorizer.pkl
  tfidf_matrix.npz
  product_profiles.csv : 17,538 rows
